# FER2013 — Colab runner

End-to-end: clone repo → download data from Kaggle → log in to Wandb → sanity checks → train all experiments → build a submission.

**Before running:** set the runtime to GPU (Runtime → Change runtime type → T4 GPU), and edit `REPO_URL` in the clone cell. See `SETUP.md` for accounts/keys.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Clone your GitHub repo
Edit `REPO_URL` to your repository.

In [ ]:
REPO_URL = "https://github.com/<you>/<repo>.git"  # <-- EDIT THIS
import os
repo = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.exists(repo):
    !git clone {REPO_URL}
%cd {repo}
!ls

## 2. Install dependencies
Colab already has torch/torchvision/pandas/sklearn; we just add wandb.

In [ ]:
!pip install -q wandb pyyaml

## 3. Download the FER2013 data from Kaggle
Upload your `kaggle.json` (Kaggle → Settings → API → Create New Token) when prompted.
Make sure you have accepted the competition rules first (see SETUP.md).

In [ ]:
from google.colab import files
print('Upload kaggle.json ...')
files.upload()
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/kaggle.json && chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
COMP = 'challenges-in-representation-learning-facial-expression-recognition-challenge'
!kaggle competitions download -c {COMP} -p data

In [ ]:
import os, glob, zipfile, tarfile
for z in glob.glob('data/*.zip'):
    with zipfile.ZipFile(z) as zf:
        zf.extractall('data')
for t in glob.glob('data/*.tar.gz'):
    with tarfile.open(t) as tf:
        tf.extractall('data')

def _find(name):
    hits = glob.glob(f'data/**/{name}', recursive=True)
    return hits[0] if hits else None

TRAIN_CSV = _find('train.csv')
TEST_CSV = _find('test.csv')

if TRAIN_CSV is None:  # fallback: build splits from fer2013.csv (has a Usage column)
    import pandas as pd
    fer = _find('fer2013.csv')
    df = pd.read_csv(fer)
    df[df.Usage == 'Training'][['emotion', 'pixels']].to_csv('data/train.csv', index=False)
    df[df.Usage != 'Training'][['pixels']].to_csv('data/test.csv', index=False)
    TRAIN_CSV, TEST_CSV = 'data/train.csv', 'data/test.csv'

print('train:', TRAIN_CSV)
print('test :', TEST_CSV)
print('files:', os.listdir('data'))

## 4. Log in to Wandb
Paste your API key from https://wandb.ai/authorize when prompted.

In [ ]:
import os
import wandb
wandb.login()
os.environ['WANDB_PROJECT'] = 'fer2013'
# os.environ['WANDB_ENTITY'] = '<your-wandb-username>'  # optional

## 5. Sanity checks (forward / backward)
Initial loss ≈ ln(7), overfit a single batch, and gradient-flow check — then a short 5-epoch run.

In [ ]:
!python -m src.train --config configs/02_tiny_cnn.yaml --train-csv {TRAIN_CSV} --run-sanity --epochs 5 --tag sanity --wandb-project fer2013

## 6. Run all experiments
Each becomes its own Wandb run (grouped by architecture). Set `QUICK_EPOCHS` for a fast smoke test.

In [ ]:
EXPERIMENTS = [
    '01_mlp',
    '02_tiny_cnn',
    '03_vgg_plain',
    '04_vgg_reg',
    '05_resnet_mini',
    '06_resnet18_transfer',
]
QUICK_EPOCHS = None  # e.g. 5 for a smoke test; None = use each config's epochs

for name in EXPERIMENTS:
    print('\n' + '=' * 70 + f'\nRUNNING {name}\n' + '=' * 70)
    extra = f'--epochs {QUICK_EPOCHS}' if QUICK_EPOCHS else ''
    !python -m src.train --config configs/{name}.yaml --train-csv {TRAIN_CSV} --test-csv {TEST_CSV} --wandb-project fer2013 {extra}

## 7. (Optional) Hyperparameter sweep
Bayesian search on the regularized VGG. Uncomment to run.

In [ ]:
# !wandb sweep configs/sweep_vgg.yaml
# Copy the printed sweep id, then run an agent for N trials:
# !wandb agent <entity>/fer2013/<sweep_id> --count 15

## 8. Build a Kaggle submission

In [ ]:
!python -m src.predict --checkpoint outputs/04_vgg_reg/best.pth --test-csv {TEST_CSV} --out outputs/submission.csv
from google.colab import files
files.download('outputs/submission.csv')

# Optional: submit directly (late submission)
# COMP = 'challenges-in-representation-learning-facial-expression-recognition-challenge'
# !kaggle competitions submit -c {COMP} -f outputs/submission.csv -m 'vgg_reg'